# unlocr 在 GPU 上运行（Hugging Face 原生模型 + FastAPI 后端）

使用 `unlocr` CLI 配合本地启动的 **FastAPI** 接口服务，在 **免费的 Google Colab T4 GPU** 运行时上，驱动原生未量化的 `baidu/Unlimited-OCR` 模型进行 OCR 解析。

### 为什么选择此路径（Hugging Face 原生 + FastAPI）：
1. **对免费版 Colab T4 友好**：vLLM 的多进程设计和缓存策略需要极高的系统内存（~16 GB+），这在 Colab 默认的 12.7 GB 内存 CPU 实例上极易导致 Out-Of-Memory (OOM) 崩溃。而通过 PyTorch 和 Hugging Face 直接加载原生模型，仅需消耗 **~6 GB 系统内存** 和 **~6 GB 显存 (VRAM)**。
2. **完整的 FP16 精度质量**：运行未量化的原生模型权重（`baidu/Unlimited-OCR` FP16），而不是 GGUF 量化版本，从而保留最高的文本和排版还原度。
3. **免去 vLLM 依赖冲突烦恼**：避开了下载 nightly vLLM 编译版本以及自定义 C++ 扩展所带来的环境脆弱性。
4. **类似于 GUI 的操作体验**：直接通过 Google Colab 的交互式表单控件，完成 PDF 上传、参数配置、运行 OCR 和结果下载。

**在开始之前：** 请确保已勾选 GPU 运行时，在顶部菜单中选择：**代码执行程序 > 更改运行时类型 > T4 GPU**（或 L4 / A100，若有可用额度）。

### 管道运行步骤：
1. **验证环境与硬件**
2. **安装基础依赖库**（poppler-utils, transformers, accelerate, fastapi, uvicorn, timm, einops, addict）
3. **从最新的工作区源码编译 `unlocr` CLI**（包含您当前所有本地及未提交的代码更改）
4. **启动本地 FastAPI 后端服务器**（在 PyTorch 中加载 Hugging Face 模型，并暴露兼容 OpenAI 的 `/v1/chat/completions` API 接口）
5. **交互式文件上传与配置**
6. **运行 OCR 任务**（使用 `unlocr` 的 `--endpoint` 模式指向本地 FastAPI 服务）
7. **下载 OCR 转换后的 Markdown 文件**

In [ ]:
# 1. 验证 GPU 是否正确挂载 (若未启用 GPU 运行时，此步骤将报错提示)
!nvidia-smi || (echo '未检测到 GPU。请在顶部菜单选择：代码执行程序 > 更改运行时类型 > 选择 GPU 硬件加速 (T4/L4/A100)。' && false)

In [ ]:
# 2. 安装系统依赖：poppler-utils (提供 pdftoppm 工具将 PDF 页面栅格化为图像)
!apt-get -qq update && apt-get -qq install -y poppler-utils build-essential git
!pdftoppm -v

In [ ]:
# 3. 从源码编译 unlocr CLI（编译时会自动提取您当前所有的本地或未提交修改）
import os, shutil

# 检查是否正处于 unlocr 工作区目录中
if os.path.exists("Cargo.toml") and os.path.exists("src/main.rs"):
    print("检测到当前正处于 unlocr 工作目录中。")
    REPO_DIR = os.getcwd()
else:
    if not os.path.exists("/content/unlocr"):
        print("正在克隆 unlocr 仓库...")
        !git clone https://github.com/whit3rabbit/unlocr.git /content/unlocr
    REPO_DIR = "/content/unlocr"

# 如果没有 cargo，则自动安装 Rust 工具链
if not shutil.which("cargo"):
    print("正在安装 Rust 编译工具链...")
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    os.environ["PATH"] = f"{os.environ['HOME']}/.cargo/bin:{os.environ['PATH']}"

print("正在以 Release 模式编译 unlocr CLI...")
%cd {REPO_DIR}
!cargo build --release
UNLOCR_BIN = os.path.join(REPO_DIR, "target/release/unlocr")
print(f"\n[成功] unlocr CLI 编译完成，程序路径: {UNLOCR_BIN}")
!{UNLOCR_BIN} --version

In [ ]:
# 4. 安装原生的 Hugging Face 与 FastAPI 服务相关 Python 依赖库
print("正在安装 Python 依赖组件...")
!pip install -q fastapi uvicorn einops timm addict transformers accelerate torchvision torch pillow

In [ ]:
#@title 5. 启动搭载 HF 模型的 FastAPI 服务器 { display-mode: "form" }
#@markdown 启动本地 FastAPI 服务器（兼容 OpenAI API 规范）。这将创建并运行 `/content/colab_server.py`，后台会自动下载未量化的原生权重模型（大小约 6.7 GB），并在端口 8000 上提供 OCR 服务。

MODEL_REPO = "baidu/Unlimited-OCR" #@param ["baidu/Unlimited-OCR", "deepseek-ai/DeepSeek-OCR"] {allow-input: true}
dtype = "float16" #@param ["float16", "bfloat16"]

server_code = """
import os
import torch
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import uvicorn
import base64
from transformers import AutoModel, AutoTokenizer

app = FastAPI()

MODEL_NAME = "{MODEL_REPO}"
torch_dtype = torch.float16 if "{dtype}" == "float16" else torch.bfloat16

print("正在从 Hugging Face 加载模型和分词器...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch_dtype
).eval().cuda()
print("模型加载成功！")

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    messages = body.get("messages", [])
    
    prompt = "document parsing."
    image_bytes = None
    
    for msg in messages:
        if msg.get("role") == "user":
            content = msg.get("content")
            if isinstance(content, list):
                for part in content:
                    if part.get("type") == "text":
                        prompt = part.get("text")
                    elif part.get("type") == "image_url":
                        img_url = part.get("image_url", {}).get("url", "")
                        if img_url.startswith("data:image/"):
                            base64_data = img_url.split(";base64,")[-1]
                            image_bytes = base64.b64decode(base64_data)
            elif isinstance(content, str):
                prompt = content

    if not image_bytes:
        return JSONResponse({"error": "No image provided"}, status_code=400)
    
    temp_img_path = "temp_page.png"
    with open(temp_img_path, "wb") as f:
        f.write(image_bytes)
        
    output_dir = "temp_out"
    os.makedirs(output_dir, exist_ok=True)
    
    # 清理历史输出
    for f in os.listdir(output_dir):
        try:
            os.remove(os.path.join(output_dir, f))
        except Exception:
            pass
        
    try:
        full_prompt = prompt if prompt.startswith("<image>") else f"<image>{prompt}"
        print(f"开始推理，提示词: {full_prompt}")
        res = model.infer(
            tokenizer,
            prompt=full_prompt,
            image_file=temp_img_path,
            output_path=output_dir,
            base_size=1024,
            image_size=1024,
            crop_mode=False,
            save_results=True
        )
        
        pred_file = os.path.join(output_dir, "pred.txt")
        if os.path.exists(pred_file):
            with open(pred_file, "r", encoding="utf-8") as f:
                ocr_text = f.read()
        else:
            ocr_text = str(res)
    except Exception as e:
        print("推理错误:", e)
        return JSONResponse({"error": str(e)}, status_code=500)
        
    response_data = {
        "choices": [
            {
                "message": {
                    "role": "assistant",
                    "content": ocr_text
                },
                "finish_reason": "stop"
            }
        ]
    }
    return JSONResponse(response_data)

@app.get("/v1/models")
async def list_models():
    return {"data": [{"id": MODEL_NAME}]}

@app.get("/health")
async def health():
    return {"status": "ok"}

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)
""".replace("{MODEL_REPO}", MODEL_REPO).replace("{dtype}", dtype)

with open("/content/colab_server.py", "w") as f:
    f.write(server_code)

import subprocess
import time
import urllib.request
import os

print("正在后台启动原生 HF FastAPI 本地服务...")
log_path = "/content/hf_server.log"
with open(log_path, "w") as log_file:
    process = subprocess.Popen(
        ["python3", "/content/colab_server.py"],
        stdout=log_file,
        stderr=log_file,
        preexec_fn=os.setsid
    )

print("正在等待服务初始化完成（模型权重下载需耗时 2-4 分钟）...")
ready = False
for i in range(120): # 最多等待 10 分钟
    try:
        urllib.request.urlopen("http://localhost:8000/health")
        ready = True
        break
    except Exception:
        time.sleep(5)
        if i % 6 == 0:
            print(f"  仍在加载中... (已过去 {i*5} 秒)")

if ready:
    print("\n[成功] 原生 Hugging Face API 后端启动成功，可以开始接收 OCR 请求！")
else:
    print("\n[错误] 后端服务未能成功启动。以下是 hf_server.log 的最后 50 行报错信息：")
    with open(log_path) as f:
        lines = f.readlines()
        print("".join(lines[-50:]))

## 6. 类似于 GUI 的交互式 OCR 转换界面

请使用下方的表单控件配置参数，并上传需要处理的文件。运行后可直接一键下载得到的 Markdown 文件。

In [ ]:
#@title 上传 PDF 并配置参数 { display-mode: "form" }
#@markdown 选择类似于桌面 GUI 客户端的 OCR 配置参数项：

task = "markdown" #@param ["markdown", "grounding", "free", "figure"]
custom_prompt = "" #@param {type:"string"}
pages = "all" #@param {type:"string"}
dpi = 144 #@param {type:"integer"}
password = "" #@param {type:"string"}
output_mode = "single" #@param ["single", "pages", "both"]

import os
import glob
import shutil
from google.colab import files

INPUT_DIR = "/content/inputs"
OUTPUT_DIR = "/content/outputs"

# 重置并清理对应文件夹
if os.path.exists(INPUT_DIR):
    shutil.rmtree(INPUT_DIR)
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("请选择一个或多个 PDF / 图像文件进行上传：")
uploaded = files.upload()
if not uploaded:
    print("未上传任何文件！")
else:
    for name in uploaded:
        dest = os.path.join(INPUT_DIR, os.path.basename(name))
        shutil.move(name, dest)
        print(f"  已暂存: {dest}")

In [ ]:
#@title 运行 OCR { display-mode: "form" }
#@markdown 开始调用已编译的 `unlocr` 工具，处理 `/content/inputs` 中的所有文件，并将最终导出的结果存入 `/content/outputs`。

import subprocess
import glob
import os

input_files = glob.glob(f"{INPUT_DIR}/*")

if not input_files:
    print("[错误] 未在 /content/inputs 目录下找到任何文件，请在前一个单元格中先上传文件。")
else:
    # 拼装命令行参数
    cmd = [
        UNLOCR_BIN,
        *input_files,
        "--endpoint", "http://localhost:8000/v1",
        "--endpoint-model", MODEL_REPO,
        "--out", OUTPUT_DIR,
        "--output-mode", output_mode,
        "--dpi", str(dpi)
    ]
    
    if task != "markdown":
        cmd += ["--task", task]
    if custom_prompt:
        cmd += ["--prompt", custom_prompt]
    if pages and pages.strip().lower() != "all":
        cmd += ["--pages", pages.strip()]
    if password:
        cmd += ["--password", password]
        
    print("执行的 OCR 命令行如下:")
    print(" ", " ".join(cmd))
    print("\n开始处理中，请稍候...\n")
    
    # 启动进程并实时流式传输输出日志
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    while True:
        output = process.stdout.readline()
        if output == '' and process.poll() is not None:
            break
        if output:
            print(output.strip())
            
    rc = process.poll()
    if rc == 0:
        print("\n[成功] OCR 处理完毕！最终文档已保存至 /content/outputs。")
        
        # 在页面上预览生成的第一个 Markdown 文件内容
        out_files = os.listdir(OUTPUT_DIR)
        if out_files:
            preview_path = os.path.join(OUTPUT_DIR, sorted(out_files)[0])
            if os.path.isfile(preview_path):
                print(f"\n--- 生成的第一个文件 {os.path.basename(preview_path)} 的内容预览（仅展示前 1000 字符） ---")
                try:
                    with open(preview_path, 'r', encoding='utf-8') as f:
                        print(f.read(1000))
                except Exception as e:
                    print(f"未能加载预览: {e}")
    else:
        print(f"\n[错误] OCR 运行失败，返回错误码: {rc}")

In [ ]:
#@title 下载 OCR 结果 { display-mode: "form" }
#@markdown 下载生成的文件。若有多个文件，会自动打包成名为 `unlocr_results.zip` 的压缩包进行下载。

import shutil
from google.colab import files
import os

out_files = os.listdir(OUTPUT_DIR)
if not out_files:
    print("没有在 /content/outputs 中找到任何需要下载的成果。")
else:
    # 如果仅有一个输出文件，则直接下载该文件
    first_path = os.path.join(OUTPUT_DIR, out_files[0])
    if len(out_files) == 1 and os.path.isfile(first_path):
        print(f"正在下载文件: {out_files[0]}...")
        files.download(first_path)
    else:
        print("正在打包成 zip 并开始下载...")
        zip_path = "/content/unlocr_results"
        shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
        files.download(f"{zip_path}.zip")